<a href="https://colab.research.google.com/github/hafsaaslam1212-star/northstar-analytics/blob/main/01_sql_in_r_ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pandas numpy matplotlib seaborn pymongo

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 58.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 25.7 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
os.listdir('/content/drive/MyDrive/northstar_data')

['northstar_dataset']

In [ ]:
import os
os.listdir('/content/drive/MyDrive/northstar_data/northstar_dataset')

['complaints.csv',
 'README.txt',
 'deliveries.csv',
 'orders.csv',
 'app_events.csv',
 'customers.csv',
 'data_dictionary.csv',
 'vehicles.csv',
 'drivers.csv',
 'incidents.csv',
 'hubs.csv']

In [ ]:
for name, df in {
    "orders": orders,
    "deliveries": deliveries,
    "customers": customers,
    "drivers": drivers,
    "vehicles": vehicles,
    "hubs": hubs
}.items():
    print(f"\n{name}")
    print(df.head())
    print(df.info())


orders
  order_id customer_id service_type    order_created_at  \
0   O00001       C0292    Passenger 2024-08-20 14:43:00   
1   O00002       C0459    Passenger 2024-05-14 22:16:00   
2   O00003       C0161    Passenger 2025-09-02 14:37:00   
3   O00004       C0520       Parcel 2025-01-11 17:15:00   
4   O00005       C0558       Retail 2025-02-17 19:32:00   

   promised_window_hours pickup_zone dropoff_zone priority_level  order_value  \
0                      6     Airport        South         Medium       126.65   
1                     24       North      AIRPORT            Low       109.30   
2                      4        West      AIRPORT           High        33.50   
3                      2   RiverSide        North         Medium        10.04   
4                     12   Riverside        SOUTH            Low       125.58   

  booking_channel  special_handling_flag  
0             App                      0  
1             App                      0  
2           Phone    

In [ ]:
#Data understanding
str(orders)
str(deliveries)
str(customers)

'data.frame':	1250 obs. of  11 variables:
 $ order_id             : chr  "O00001" "O00002" "O00003" "O00004" ...
 $ customer_id          : chr  "C0292" "C0459" "C0161" "C0520" ...
 $ service_type         : chr  "Passenger" "Passenger" "Passenger" "Parcel" ...
 $ order_created_at     : chr  "2024-08-20 14:43:00" "2024-05-14 22:16:00" "2025-09-02 14:37:00" "2025-01-11 17:15:00" ...
 $ promised_window_hours: int  6 24 4 2 12 1 2 4 12 6 ...
 $ pickup_zone          : chr  "Airport" "North" "West" "RiverSide" ...
 $ dropoff_zone         : chr  "South" "AIRPORT" "AIRPORT" "North" ...
 $ priority_level       : chr  "Medium" "Low" "High" "Medium" ...
 $ order_value          : num  126.7 109.3 33.5 10 125.6 ...
 $ booking_channel      : chr  "App" "App" "Phone" "App" ...
 $ special_handling_flag: int  0 0 0 1 0 1 0 0 0 0 ...
'data.frame':	950 obs. of  13 variables:
 $ delivery_id                  : chr  "DL00001" "DL00002" "DL00003" "DL00004" ...
 $ order_id                     : chr  "O00938" "

In [ ]:
#HANDLE MISSING VALUES
colSums(is.na(orders))
colSums(is.na(deliveries))
colSums(is.na(customers))

order_id           customer_id          service_type 
                    0                     0                     0 
     order_created_at promised_window_hours           pickup_zone 
                    0                     0                     0 
         dropoff_zone        priority_level           order_value 
                    0                     0                     0 
      booking_channel special_handling_flag 
                    0                     0

delivery_id                      order_id 
                            0                             0 
                    driver_id                    vehicle_id 
                            0                             0 
                       hub_id                 dispatch_time 
                            0                             0 
        delivery_completed_at               delivery_status 
                            0                             0 
            route_distance_km   manual_route_override_count 
                            0                             0 
  proof_of_completion_missing customer_rating_post_delivery 
                            0                            14 
          fuel_or_charge_cost 
                            0

customer_id                  age            home_zone 
                   0                    0                    0 
       customer_type          signup_date        loyalty_score 
                   0                    0                   20 
app_engagement_score    preferred_channel       account_status 
                   0                    0                    0

In [ ]:
install.packages("stringr")
library(stringr)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)



In [ ]:
#Remove Duplicates
orders <- distinct(orders)
deliveries <- distinct(deliveries)
customers <- distinct(customers)

In [ ]:
#Quality check
cat("Orders rows:", nrow(orders), "\n")
cat("Deliveries rows:", nrow(deliveries), "\n")
cat("Customers rows:", nrow(customers), "\n")

Orders rows: 1250 
Deliveries rows: 950 
Customers rows: 650 


In [ ]:
clean_deliveries <- deliveries %>%
  filter(!is.na(delivery_status),
         !is.na(route_distance_km),
         route_distance_km > 0)

In [ ]:
#STANDARDISE TEXT FIELDS
orders$pickup_zone      <- str_to_title(trimws(orders$pickup_zone))
orders$dropoff_zone     <- str_to_title(trimws(orders$dropoff_zone))

customers$home_zone     <- str_to_title(trimws(customers$home_zone))
drivers$base_zone       <- str_to_title(trimws(drivers$base_zone))
vehicles$assigned_zone  <- str_to_title(trimws(vehicles$assigned_zone))
hubs$zone               <- str_to_title(trimws(hubs$zone))

deliveries$delivery_status <- str_to_title(trimws(deliveries$delivery_status))
complaints$severity <- str_to_title(trimws(complaints$severity))

### **SQL IN R ANALYTICS**

In [ ]:
install.packages("sqldf")
install.packages("dplyr")
install.packages("ggplot2")

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependency ‘RSQLite’


Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)



In [ ]:
library(sqldf)
library(dplyr)
library(ggplot2)

Loading required package: gsubfn

Loading required package: proto

Warning message:
“no DISPLAY variable so Tk is not available”
Loading required package: RSQLite


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union




In [ ]:
BASE = '/content/drive/MyDrive/northstar_data/northstar_dataset/'

In [ ]:
system("pip install -q pandas")


In [ ]:
orders <- read.csv("/content/drive/MyDrive/northstar_data/northstar_dataset/orders.csv")
deliveries <- read.csv("/content/drive/MyDrive/northstar_data/northstar_dataset/deliveries.csv")
customers <- read.csv("/content/drive/MyDrive/northstar_data/northstar_dataset/customers.csv")
drivers <- read.csv("/content/drive/MyDrive/northstar_data/northstar_dataset/drivers.csv")
vehicles <- read.csv("/content/drive/MyDrive/northstar_data/northstar_dataset/vehicles.csv")
hubs <- read.csv("/content/drive/MyDrive/northstar_data/northstar_dataset/hubs.csv")
incidents <- read.csv("/content/drive/MyDrive/northstar_data/northstar_dataset/incidents.csv")
complaints <- read.csv("/content/drive/MyDrive/northstar_data/northstar_dataset/complaints.csv")
app_events <- read.csv("/content/drive/MyDrive/northstar_data/northstar_dataset/app_events.csv")

In [ ]:
list.files("/content/drive/MyDrive/northstar_data/northstar_dataset/")

[1] "app_events.csv"      "complaints.csv"      "customers.csv"      
 [4] "data_dictionary.csv" "deliveries.csv"      "drivers.csv"        
 [7] "hubs.csv"            "incidents.csv"       "orders.csv"         
[10] "README.txt"          "vehicles.csv"

In [ ]:
sqldf("SELECT * FROM orders LIMIT 5")

order_id,customer_id,service_type,order_created_at,promised_window_hours,pickup_zone,dropoff_zone,priority_level,order_value,booking_channel,special_handling_flag
<chr>,<chr>,<chr>,<chr>,<int>,<chr>,<chr>,<chr>,<dbl>,<chr>,<int>
O00001,C0292,Passenger,2024-08-20 14:43:00,6,Airport,South,Medium,126.65,App,0
O00002,C0459,Passenger,2024-05-14 22:16:00,24,North,AIRPORT,Low,109.30,App,0
O00003,C0161,Passenger,2025-09-02 14:37:00,4,West,AIRPORT,High,33.50,Phone,0
O00004,C0520,Parcel,2025-01-11 17:15:00,2,RiverSide,North,Medium,10.04,App,1
O00005,C0558,Retail,2025-02-17 19:32:00,12,Riverside,SOUTH,Low,125.58,Phone,0


In [ ]:
#Delivery Status Distribution Analysis
sqldf("
SELECT delivery_status,
       COUNT(*) AS total_deliveries
FROM deliveries
GROUP BY delivery_status
ORDER BY total_deliveries DESC
")

delivery_status,total_deliveries
<chr>,<int>
Ontime,616
Delayed,202
Failed,132


In [ ]:
#Hub Performance Analysis
sqldf("
SELECT h.hub_name,
       h.zone,
       d.delivery_status,
       COUNT(*) AS count
FROM deliveries d
JOIN hubs h ON d.hub_id = h.hub_id
GROUP BY h.hub_name, h.zone, d.delivery_status
ORDER BY h.hub_name
")

hub_name,zone,delivery_status,count
<chr>,<chr>,<chr>,<int>
Airport Hub,Airport,Delayed,27
Airport Hub,Airport,Failed,15
Airport Hub,Airport,Ontime,62
Central Core,Central,Delayed,25
Central Core,Central,Failed,23
Central Core,Central,Ontime,67
East Dock,East,Delayed,23
East Dock,East,Failed,11
East Dock,East,Ontime,85


In [ ]:
#Driver Behaviour Analysis
sqldf("
SELECT driver_id,
       COUNT(*) AS total_deliveries,
       AVG(manual_route_override_count) AS avg_overrides
FROM deliveries
GROUP BY driver_id
ORDER BY avg_overrides DESC
LIMIT 10
")

driver_id,total_deliveries,avg_overrides
<chr>,<int>,<dbl>
D112,2,4.500000
D127,6,2.833333
D021,2,2.500000
D139,5,2.000000
D130,8,2.000000
D124,4,2.000000
D105,7,2.000000
D085,4,2.000000
D079,2,2.000000


In [ ]:
#Customer Service Failure Analysis
sqldf("
SELECT c.customer_id,
       COUNT(DISTINCT cp.complaint_id) AS complaint_count,
       COUNT(DISTINCT d.delivery_id) AS deliveries
FROM customers c
LEFT JOIN complaints cp ON c.customer_id = cp.customer_id
LEFT JOIN orders o ON c.customer_id = o.customer_id
LEFT JOIN deliveries d ON o.order_id = d.order_id
GROUP BY c.customer_id
HAVING complaint_count >= 2
ORDER BY complaint_count DESC
")

customer_id,complaint_count,deliveries
<chr>,<int>,<int>
C0368,4,3
C0626,3,2
C0573,3,1
C0545,3,5
C0421,3,3
C0372,3,4
C0282,3,2
C0242,3,3
C0191,3,1


In [ ]:
#Vehicle Performance and Incident Analysis
sqldf("
SELECT
  CASE
    WHEN v.battery_health_pct >= 80 THEN 'Good'
    WHEN v.battery_health_pct >= 60 THEN 'Fair'
    ELSE 'Poor'
  END AS battery_status,
  COUNT(i.incident_id) AS incidents
FROM vehicles v
JOIN deliveries d ON v.vehicle_id = d.vehicle_id
LEFT JOIN incidents i ON d.delivery_id = i.delivery_id
GROUP BY battery_status
")

battery_status,incidents
<chr>,<int>
Fair,121
Good,115
Poor,44
